# 1.- HEADER
1. Header fijo (256 bytes)
2. Header por canal (256 bytes × nº canales)


## 1.1 HEADER FIJO (PRIMEROS 256 BYTES)


| Bytes   | Tamaño | Significado        |
| ------- | ------ | ------------------ |
| 0–7     | 8      | versión            |
| 8–87    | 80     | patient ID         |
| 88–167  | 80     | recording ID       |
| 168–175 | 8      | fecha              |
| 176–183 | 8      | hora               |
| 184–191 | 8      | tamaño header      |
| 192–235 | 44     | reservado          |
| 236–243 | 8      | nº data records    |
| 244–251 | 8      | duración de record |
| 252–255 | 4      | nº canales         |


### Lectura manual del header (nivel binario)

Vamos a leer directamente los primeros bytes del archivo `.bdf` para entender su estructura interna.

Esto nos permite verificar cómo está organizado el archivo según el estándar EDF/BDF.

In [27]:
from pathlib import Path
from typing import BinaryIO

file_path: Path = Path("dataset/s01.bdf")

with open(file_path, "rb") as f:
    header: bytes = f.read(520)

print(f"Tamaño del header leído: {len(header)} bytes")

Tamaño del header leído: 520 bytes


In [28]:
from typing import Tuple

def read_str(data: bytes, start: int, length: int) -> str:
    return data[start : start + length].decode("ascii", errors="ignore").strip()

In [29]:
version: str = read_str(header, 0, 8)
patient_id: str = read_str(header, 8, 80)
recording_id: str = read_str(header, 88, 80)
start_date: str = read_str(header, 168, 8)
start_time: str = read_str(header, 176, 8)
header_bytes: str = read_str(header, 184, 8)
reserved: str = read_str(header, 192, 44)
num_records: str = read_str(header, 236, 8)
record_duration: str = read_str(header, 244, 8)
num_channels: str = read_str(header, 252, 4)

print("Versión:", version)
print("Paciente:", patient_id)
print("Recording:", recording_id)
print("Fecha:", start_date)
print("Hora:", start_time)
print("Header bytes:", header_bytes)
print("Reservado:", reserved)
print("N° records:", num_records)
print("Duración record:", record_duration)
print("N° canales:", num_channels)

header_bytes: int = int(header_bytes)
num_channels:int = int(num_channels)

Versión: BIOSEMI
Paciente: s01
Recording: 
Fecha: 01.07.10
Hora: 10.00.16
Header bytes: 12544
Reservado: 24BIT
N° records: 3869
Duración record: 1
N° canales: 48


## 1.2 HEADER POR CANAL (LO MÁS INTERESANTE)
- cada canal tiene 256 bytes

| Campo              | Tamaño |
| ------------------ | ------ |
| label              | 16     |
| transducer         | 80     |
| physical dimension | 8      |
| physical min       | 8      |
| physical max       | 8      |
| digital min        | 8      |
| digital max        | 8      |
| prefiltering       | 80     |
| samples per record | 8      |
| reserved           | 32     |


### Lectura correcta del header completo

El archivo `.bdf` tiene:

1. Header general: 256 bytes.
2. Header de señales/canales: 256 bytes × número de canales.

Pero los datos de los canales no están almacenados como:

```text
canal 1 completo, canal 2 completo, canal 3 completo...
```
sino por bloques:

- labels de todos los canales
- transducers de todos los canales
- unidades físicas de todos los canales
- mínimos físicos de todos los canales
- máximos físicos de todos los canales
- ....
- ....
- ....
- reserved de todos los canales


In [30]:
with open(file_path, "rb") as file:
    full_header: bytes = file.read(int(header_bytes))

print(f"Bytes leídos del header completo: {len(full_header)}")

Bytes leídos del header completo: 12544


### Extraer nombres de canales

Después de los primeros 256 bytes, vienen los nombres de todos los canales.

Cada nombre ocupa 16 bytes.

Como tenemos 48 canales:

```text
48 × 16 = 768 bytes

In [31]:
labels_start: int = 256
label_size: int = 16

channel_labels: list[str] = [
    read_str(full_header, labels_start + channel_index * label_size, label_size)
    for channel_index in range(int(num_channels))
]

for index, label in enumerate(channel_labels, start=1):
    print(f"{index:02d}. {label}")

01. Fp1
02. AF3
03. F7
04. F3
05. FC1
06. FC5
07. T7
08. C3
09. CP1
10. CP5
11. P7
12. P3
13. Pz
14. PO3
15. O1
16. Oz
17. O2
18. PO4
19. P4
20. P8
21. CP6
22. CP2
23. C4
24. T8
25. FC6
26. FC2
27. F4
28. F8
29. AF4
30. Fp2
31. Fz
32. Cz
33. EXG1
34. EXG2
35. EXG3
36. EXG4
37. EXG5
38. EXG6
39. EXG7
40. EXG8
41. GSR1
42. GSR2
43. Erg1
44. Erg2
45. Resp
46. Plet
47. Temp
48. Status


In [32]:
transducer_size: int = 80
physical_dimension_size: int = 8

transducers_start: int = labels_start + (num_channels * label_size)
physical_dimensions_start: int = transducers_start + (num_channels * transducer_size)

physical_dimensions: list[str] = [
    read_str(
        full_header,
        physical_dimensions_start + channel_index * physical_dimension_size,
        physical_dimension_size,
    )
    for channel_index in range(num_channels)
]

for label, unit in zip(channel_labels, physical_dimensions):
    print(f"{label}: {unit}")

Fp1: uV
AF3: uV
F7: uV
F3: uV
FC1: uV
FC5: uV
T7: uV
C3: uV
CP1: uV
CP5: uV
P7: uV
P3: uV
Pz: uV
PO3: uV
O1: uV
Oz: uV
O2: uV
PO4: uV
P4: uV
P8: uV
CP6: uV
CP2: uV
C4: uV
T8: uV
FC6: uV
FC2: uV
F4: uV
F8: uV
AF4: uV
Fp2: uV
Fz: uV
Cz: uV
EXG1: uV
EXG2: uV
EXG3: uV
EXG4: uV
EXG5: uV
EXG6: uV
EXG7: uV
EXG8: uV
GSR1: nS
GSR2: nS
Erg1: uV
Erg2: uV
Resp: uV
Plet: uV
Temp: Celsius
Status: Boolean


___

### Lectura completa de los headers por canal

En EDF/BDF, después del header fijo de 256 bytes, aparecen los campos de todos los canales agrupados por tipo de campo.

El orden es:

1. `label` → 16 bytes por canal
2. `transducer` → 80 bytes por canal
3. `physical_dimension` → 8 bytes por canal
4. `physical_min` → 8 bytes por canal
5. `physical_max` → 8 bytes por canal
6. `digital_min` → 8 bytes por canal
7. `digital_max` → 8 bytes por canal
8. `prefiltering` → 80 bytes por canal
9. `samples_per_record` → 8 bytes por canal
10. `reserved` → 32 bytes por canal

Como tenemos 48 canales, cada bloque tiene `tamaño_del_campo × 48` bytes.

### Definir estructura de campos del header por canal

Ahora definimos el nombre y tamaño de cada campo. Luego calculamos automáticamente dónde empieza cada bloque dentro del header completo.

In [33]:
from typing import Dict, List, Tuple

signal_header_start: int = 256

field_sizes: List[Tuple[str, int]] = [
    ("label", 16),
    ("transducer", 80),
    ("physical_dimension", 8),
    ("physical_min", 8),
    ("physical_max", 8),
    ("digital_min", 8),
    ("digital_max", 8),
    ("prefiltering", 80),
    ("samples_per_record", 8),
    ("reserved", 32),
]

field_starts: Dict[str, int] = {}

current_start: int = signal_header_start

for field_name, field_size in field_sizes:
    field_starts[field_name] = current_start
    current_start += field_size * num_channels

for field_name, start in field_starts.items():
    print(f"{field_name}: empieza en byte {start}")

label: empieza en byte 256
transducer: empieza en byte 1024
physical_dimension: empieza en byte 4864
physical_min: empieza en byte 5248
physical_max: empieza en byte 5632
digital_min: empieza en byte 6016
digital_max: empieza en byte 6400
prefiltering: empieza en byte 6784
samples_per_record: empieza en byte 10624
reserved: empieza en byte 11008


### Extraer todos los campos para todos los canales

Aquí construiremos una tabla donde cada fila representa un canal y cada columna representa un campo del header.

In [34]:
from typing import List, Dict, Any
import pandas as pd

channel_rows: List[Dict[str, Any]] = []

for channel_index in range(num_channels):
    row: Dict[str, Any] = {"channel_index": channel_index + 1}

    for field_name, field_size in field_sizes:
        start: int = field_starts[field_name] + channel_index * field_size
        value: str = read_str(full_header, start, field_size)
        row[field_name] = value

    channel_rows.append(row)

header_df: pd.DataFrame = pd.DataFrame(channel_rows)

header_df

,channel_index,label,transducer,physical_dimension,physical_min,physical_max,digital_min,digital_max,prefiltering,samples_per_record,reserved
0,1,Fp1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
1,2,AF3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
2,3,F7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
3,4,F3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
4,5,FC1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
5,6,FC5,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
6,7,T7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
7,8,C3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
8,9,CP1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
9,10,CP5,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON


### Imprimir la tabla completa en texto

Si la tabla se ve cortada en Jupyter, podemos imprimir cada canal de forma ordenada.

In [35]:
for _, row in header_df.iterrows():
    print("=" * 80)
    print(f"Canal {row['channel_index']:02d}: {row['label']}")
    print(f"Transducer: {row['transducer']}")
    print(f"Physical dimension: {row['physical_dimension']}")
    print(f"Physical min: {row['physical_min']}")
    print(f"Physical max: {row['physical_max']}")
    print(f"Digital min: {row['digital_min']}")
    print(f"Digital max: {row['digital_max']}")
    print(f"Prefiltering: {row['prefiltering']}")
    print(f"Samples per record: {row['samples_per_record']}")
    print(f"Reserved: {row['reserved']}")

Canal 01: Fp1
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 02: AF3
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 03: F7
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 04: F3
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 05: FC1
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
D

## Interpretación del header de canales

El archivo `s01.bdf` contiene 48 canales:

- Canales 1–32: EEG.
- Canales 33–40: EXG, señales auxiliares registradas en microvoltios.
- Canales 41–47: señales fisiológicas periféricas.
- Canal 48: `Status`, usado para triggers/eventos del experimento.

El header también indica la unidad física, el rango físico, el rango digital y el filtrado aplicado o declarado para cada canal.

Los campos `digital_min` y `digital_max` representan el rango de valores enteros almacenados en el archivo.  
Los campos `physical_min` y `physical_max` indican cómo esos valores digitales deben convertirse a unidades físicas como `uV`, `nS` o `Celsius`.